# COA Analysis — Course of Action Generator (E5.2)

This notebook demonstrates and visualises the **COA generator** implemented in
`analysis/coa_generator.py`.

> **Research question:** Which tactical archetype gives the trained policy the
> best predicted outcome against a given opponent, and how do the COAs compare
> on win rate, casualties, and terrain control?

## What this notebook does

1. Generates ≥ 5 distinct COAs using Monte-Carlo rollout.
2. Visualises the **composite score, win rate, and casualty efficiency** for each COA.
3. Plots the **action-profile heatmap** (mean move/rotate/fire by episode quartile).
4. Shows a **radar chart** comparing all COAs across all scoring dimensions.

## Acceptance criteria (E5.2)

| Criterion | Check |
|-----------|-------|
| Generator produces ≥ 5 distinct COAs | Cell 3 |
| COAs differ in aggregate action sequences | Action-profile heatmap |
| REST API returns scored COA list as JSON | `api/coa_endpoint.py` |

## Usage

### With a trained checkpoint
1. Set `CHECKPOINT_PATH` to your SB3 ``.zip`` checkpoint.
2. Set `N_ROLLOUTS` to the desired number of Monte-Carlo rollouts per COA.
3. Run all cells top-to-bottom.

### Without a checkpoint (demo mode)
Leave `CHECKPOINT_PATH = None`.  The notebook uses a random policy so that
every plot renders immediately without requiring a trained model.

In [ ]:
# ── Dependencies ─────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to sys.path so local imports work from notebooks/ dir
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import matplotlib.cm as cm

from analysis.coa_generator import (
    COAGenerator,
    CourseOfAction,
    STRATEGY_LABELS,
    generate_coas,
)
from envs.battalion_env import BattalionEnv

# ── Configuration ─────────────────────────────────────────────────────────────
# Set to a path to an SB3 .zip checkpoint, or leave None to use random policy.
CHECKPOINT_PATH = None   # e.g. Path("../checkpoints/my_run/final")

N_ROLLOUTS   = 30    # Monte-Carlo rollouts per COA
N_COAS       = 5     # Number of distinct COAs to generate
SEED         = 42    # For reproducibility
OPPONENT_LEVEL = 3   # Scripted Red opponent level (1–5)

print(f"Project root  : {PROJECT_ROOT}")
print(f"Checkpoint    : {CHECKPOINT_PATH}")
print(f"n_rollouts    : {N_ROLLOUTS}")
print(f"n_coas        : {N_COAS}")
print(f"Seed          : {SEED}")
print(f"Opponent level: {OPPONENT_LEVEL}")

## 1 — Load Policy (optional)

In [ ]:
policy = None

if CHECKPOINT_PATH is not None:
    from stable_baselines3 import PPO
    policy = PPO.load(str(CHECKPOINT_PATH))
    print(f"✅ Loaded policy from {CHECKPOINT_PATH}")
else:
    print("⚠️  No checkpoint provided — using random policy (demo mode).")

## 2 — Generate COAs

In [ ]:
env = BattalionEnv(
    curriculum_level=OPPONENT_LEVEL,
    randomize_terrain=False,
    max_steps=200,
)

generator = COAGenerator(
    env=env,
    n_rollouts=N_ROLLOUTS,
    n_coas=N_COAS,
    seed=SEED,
)

import time
t0 = time.time()
coas = generator.generate(policy=policy)
elapsed = time.time() - t0

print(f"Generated {len(coas)} COAs in {elapsed:.1f}s")
print()
print(f"{'Rank':>4}  {'Label':<18}  {'Win%':>6}  {'Blue cas':>8}  {'Red cas':>8}  {'Terrain':>8}  {'Composite':>10}")
print("-" * 75)
for coa in coas:
    s = coa.score
    print(
        f"{coa.rank:>4}  {coa.label:<18}  "
        f"{s.win_rate:>6.1%}  "
        f"{s.blue_casualties:>8.3f}  "
        f"{s.red_casualties:>8.3f}  "
        f"{s.terrain_control:>8.3f}  "
        f"{s.composite:>10.4f}"
    )

## 3 — Composite Score Bar Chart

In [ ]:
labels     = [c.label for c in coas]
win_rates  = [c.score.win_rate       for c in coas]
composites = [c.score.composite      for c in coas]
blue_cas   = [c.score.blue_casualties for c in coas]
red_cas    = [c.score.red_casualties  for c in coas]
terrain    = [c.score.terrain_control for c in coas]

x = np.arange(len(labels))
width = 0.18

fig, ax = plt.subplots(figsize=(12, 5))
bars_wr  = ax.bar(x - 2*width, win_rates,  width, label="Win rate",         color="steelblue")
bars_comp= ax.bar(x - 1*width, composites, width, label="Composite score",  color="darkorange")
bars_rc  = ax.bar(x + 0*width, red_cas,    width, label="Red casualties",   color="firebrick")
bars_bc  = ax.bar(x + 1*width, blue_cas,   width, label="Blue casualties",  color="royalblue")
bars_tc  = ax.bar(x + 2*width, terrain,    width, label="Terrain control",  color="seagreen")

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score (normalised)")
ax.set_title("COA Comparison — Scoring Dimensions")
ax.legend(loc="upper right", fontsize=8)
ax.axhline(0.5, color="grey", linewidth=0.8, linestyle="--", alpha=0.5)
fig.tight_layout()
plt.show()

## 4 — Action-Profile Heatmap

Each row is a COA; each column is a (dimension, episode-quartile) pair.  Darker
cells indicate higher mean action values, showing how different COAs use distinct
movement / fire patterns across the episode.

In [ ]:
dims    = ["move", "rotate", "fire"]
quarts  = ["q1", "q2", "q3", "q4"]
col_labels = [f"{d}_{q}" for d in dims for q in quarts]

matrix = np.zeros((len(coas), len(col_labels)))
for i, coa in enumerate(coas):
    for j, col in enumerate(col_labels):
        matrix[i, j] = coa.action_summary.get(col, 0.0)

fig, ax = plt.subplots(figsize=(14, max(3, len(coas) * 0.8 + 1)))
im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=-1, vmax=1)

ax.set_xticks(np.arange(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticks(np.arange(len(coas)))
ax.set_yticklabels([f"#{c.rank} {c.label}" for c in coas])
ax.set_title("COA Action Profiles (mean action per episode quartile)")

fig.colorbar(im, ax=ax, label="Mean action value")

# Annotate cells
for i in range(len(coas)):
    for j in range(len(col_labels)):
        ax.text(j, i, f"{matrix[i, j]:.2f}",
                ha="center", va="center", fontsize=6, color="black")

fig.tight_layout()
plt.show()

## 5 — Radar Chart

Radar chart comparing each COA across all five scoring dimensions.

In [ ]:
radar_dims = ["win_rate", "red_casualties", "terrain_control",
              "1-blue_cas", "composite"]

def _get_radar_values(coa: CourseOfAction) -> list:
    s = coa.score
    return [
        s.win_rate,
        s.red_casualties,
        s.terrain_control,
        1.0 - s.blue_casualties,  # lower casualties = better
        s.composite,
    ]

N_DIM = len(radar_dims)
angles = np.linspace(0, 2 * np.pi, N_DIM, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})

colors = plt.cm.tab10(np.linspace(0, 1, len(coas)))

for coa, color in zip(coas, colors):
    values = _get_radar_values(coa)
    values += values[:1]
    ax.plot(angles, values, color=color, linewidth=2, linestyle="solid",
            label=f"#{coa.rank} {coa.label}")
    ax.fill(angles, values, color=color, alpha=0.08)

ax.set_thetagrids(np.degrees(angles[:-1]), radar_dims)
ax.set_ylim(0, 1)
ax.set_title("COA Radar Chart", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=9)
plt.tight_layout()
plt.show()

## 6 — Export COA Summary as JSON

In [ ]:
summary = {
    "n_coas": len(coas),
    "n_rollouts": N_ROLLOUTS,
    "seed": SEED,
    "opponent_level": OPPONENT_LEVEL,
    "coas": [c.as_dict() for c in coas],
}

print(json.dumps(summary, indent=2))

## 7 — REST API Demo

The same COA generator is accessible via the Flask REST API in `api/coa_endpoint.py`.
Start the server with:

```bash
python api/coa_endpoint.py
```

Then send a request (example using `requests`):

In [ ]:
# ── In-process API demo (no server needed) ──────────────────────────────────
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from api.coa_endpoint import app as _flask_app

with _flask_app.test_client() as client:
    resp = client.post("/coas", json={
        "n_rollouts": 5,
        "n_coas": 3,
        "seed": 7,
        "env_kwargs": {"curriculum_level": OPPONENT_LEVEL, "randomize_terrain": False,
                       "max_steps": 50},
    })
    api_result = resp.get_json()

print(f"HTTP status : {resp.status_code}")
print(f"COAs returned: {api_result['n_coas']}")
for c in api_result['coas']:
    print(f"  #{c['rank']} {c['label']:18s}  composite={c['score']['composite']:.4f}")

---
> 🤖 *Notebook auto-generated as part of Epic E5.2 — COA Generator.*